# 03 · Access to treatment, relative to need

The original plan was to flag "treatment deserts": tracts far from any site offering medications for opioid use disorder (MOUD). That measure ran backwards. The tracts with the most overdose deaths are the *closest* to treatment, because clinics open where the need is.

So this notebook measures access a different way, with a two-step floating catchment area (2SFCA) score: how many MOUD sites a tract can reach, divided by the demand competing for them. The method and its limitations are written up in `docs/methodology.md` (D4c).

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd

from src import plots
from src.build_tract_table import add_access_scores, label_quadrants
from src.config import CATCHMENT_SENSITIVITY_MILES, CRS_LATLON, CRS_PROJECTED, PROCESSED_DIR, REFERENCE_DIR

plots.set_style()

In [ ]:
tract_table = pd.read_csv(PROCESSED_DIR / "tract_table.csv", dtype={"GEOID": str})
tracts = gpd.read_file(REFERENCE_DIR / "cook_tracts.gpkg").merge(tract_table, on="GEOID")
facilities = pd.read_csv(PROCESSED_DIR / "treatment_facilities.csv")
moud_sites = facilities[facilities["offers_moud"]]

rated = tract_table.dropna(subset=["overdose_rate_per_100k"]).copy()
rated["burden_quartile"] = pd.qcut(rated["overdose_rate_per_100k"], 4,
                                   labels=["Lowest", "Second", "Third", "Highest"])
print(f"{len(moud_sites)} MOUD sites, {len(rated):,} tracts with a rate")

## Why distance doesn't work here

Group tracts into quartiles by overdose rate and compare three access measures. Distance and population-based access both say the hardest-hit tracts are the *best* served. Only the need-based measure shows the gap.

In [ ]:
by_quartile = rated.groupby("burden_quartile", observed=True).agg(
    tracts=("GEOID", "size"),
    median_rate=("overdose_rate_per_100k", "median"),
    miles_to_nearest_moud=("miles_to_moud", "median"),
    sites_per_100k_residents=("access_moud_per_100k_pop", "median"),
    sites_per_100_deaths=("access_moud_per_100_deaths", "median"),
).round(2)
by_quartile

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=False)
panels = [
    ("sites_per_100k_residents", "Per resident", "MOUD sites in reach per 100k residents"),
    ("sites_per_100_deaths", "Per overdose death", "MOUD sites in reach per 100 annual deaths"),
]
quartile_labels = [f"{label}\nburden" for label in by_quartile.index]
# Highlight the highest-burden quartile; the rest stay neutral
bar_colors = [plots.GRAY_MID] * 3 + [plots.ORANGE]

for ax, (column, title, ylabel) in zip(axes, panels):
    ax.bar(quartile_labels, by_quartile[column], color=bar_colors, width=0.65)
    for x, value in enumerate(by_quartile[column]):
        ax.text(x, value, f"{value:.1f}", ha="center", va="bottom", fontsize=9, color=plots.TEXT_PRIMARY)
    ax.set_title(title)
    ax.set_ylabel(ylabel)

fig.suptitle("The hardest-hit tracts look well served per resident, and underserved per death",
             x=0.01, ha="left", fontsize=13, fontweight="bold", y=1.04)
plots.add_source_note(fig, "Tracts grouped by overdose rate quartile, medians shown. 2SFCA with a 2-mile catchment. "
                           "Sources: Cook County ME, FindTreatment.gov, 2020 Census.")
fig.tight_layout()
plots.save_figure(fig, "access_two_measures.png")
plt.show()

## Where supply falls short of need

A tract is **high burden** if its overdose rate is in the county's top quartile, and **low access** if it has fewer MOUD sites per death than the county as a whole.

In [ ]:
group_order = ["high burden, low access", "high burden, high access",
               "low burden, low access", "low burden, high access"]

fig, ax = plots.map_axes()
plots.plot_categories(ax, tracts, "access_group", plots.ACCESS_GROUP_COLORS, group_order)

# Show MOUD sites inside the county so readers can see clinics sit right next to the orange tracts
sites = gpd.GeoDataFrame(moud_sites, geometry=gpd.points_from_xy(moud_sites["longitude"], moud_sites["latitude"]),
                         crs=CRS_LATLON).to_crs(tracts.crs)
sites = sites[sites.within(tracts.union_all())]
sites.plot(ax=ax, color=plots.TEXT_PRIMARY, markersize=6, edgecolor=plots.SURFACE, linewidth=0.4)

# add a marker entry to the group legend that plot_categories already drew
legend_handles = ax.get_legend().legend_handles + [plt.Line2D([], [], marker="o", linestyle="", color=plots.TEXT_PRIMARY,
                                                      markersize=4, label="MOUD site")]
ax.legend(handles=legend_handles, loc="lower left", fontsize=9)
ax.set_title("Overdose burden vs treatment supply, by census tract")
plots.add_source_note(fig, "High burden = top quartile overdose rate, 2015-2025. Low access = fewer MOUD sites per "
                           "overdose death than the county overall (2SFCA, 2-mile catchment).")
plots.save_figure(fig, "access_groups_map.png")
plt.show()

In [ ]:
groups = tract_table[tract_table["access_group"] != "no rate"].groupby("access_group")
profile = pd.DataFrame({
    "tracts": groups.size(),
    "share_of_population": groups["population_2020"].sum() / tract_table["population_2020"].sum(),
    "share_of_deaths": groups["overdose_deaths"].sum() / tract_table["overdose_deaths"].sum(),
}).join(groups[["overdose_rate_per_100k", "access_moud_per_100_deaths", "miles_to_moud",
                "median_household_income", "pct_poverty", "pct_uninsured", "pct_no_vehicle",
                "pct_nh_black", "pct_hispanic", "pct_nh_white"]].median())
profile.loc[group_order].round(2).T

## Who lives in the underserved tracts

The cleanest comparison is between the two *high burden* groups: similar overdose rates, very different treatment supply. The county median is shown for reference.

In [ ]:
measures = {
    "pct_no_vehicle": "Households without a vehicle (%)",
    "pct_poverty": "Residents below poverty (%)",
    "pct_uninsured": "Residents uninsured (%)",
    "pct_nh_black": "Black residents (%)",
}
comparison = pd.DataFrame({
    "High burden, low access": profile.loc["high burden, low access", list(measures)],
    "High burden, high access": profile.loc["high burden, high access", list(measures)],
    "County (all tracts)": tract_table[list(measures)].median(),
})

fig, axes = plt.subplots(1, len(measures), figsize=(12, 3.6))
series_colors = [plots.ORANGE, plots.BLUE, plots.GRAY_MID]
for ax, (column, title) in zip(axes, measures.items()):
    values = comparison.loc[column]
    ax.bar(range(3), values, color=series_colors, width=0.7)
    for x, value in enumerate(values):
        ax.text(x, value, f"{value:.0f}", ha="center", va="bottom", fontsize=9)
    ax.set_title(title, fontsize=10)
    ax.set_xticks([])
    ax.set_ylim(0, max(values) * 1.2)

handles = [plt.Rectangle((0, 0), 1, 1, color=color) for color in series_colors]
fig.legend(handles, comparison.columns, loc="lower center", ncol=3, bbox_to_anchor=(0.5, -0.1))
fig.suptitle("Underserved high-burden tracts are poorer and more car-free than better-served ones",
             x=0.01, ha="left", fontsize=13, fontweight="bold", y=1.05)
fig.tight_layout()
# the legend sits along the bottom, so push the source note below it
plots.add_source_note(fig, "Medians across tracts. Source: ACS 2019-2023 5-year estimates.", y=-0.14)
plots.save_figure(fig, "underserved_profile.png")
plt.show()

## Does the result depend on the catchment size?

Rerun the whole access calculation at 1, 2, and 3 miles and check how much the underserved group changes.

In [ ]:
centers = gpd.read_file(REFERENCE_DIR / "tract_population_centers.gpkg")
all_sites = gpd.GeoDataFrame(facilities, geometry=gpd.points_from_xy(facilities["longitude"], facilities["latitude"]),
                             crs=CRS_LATLON).to_crs(CRS_PROJECTED)

underserved_by_catchment = {}
rows = []
for miles in CATCHMENT_SENSITIVITY_MILES:
    scored = label_quadrants(add_access_scores(tract_table, centers, all_sites, miles))
    underserved = scored[scored["access_group"] == "high burden, low access"]
    underserved_by_catchment[miles] = set(underserved["GEOID"])
    rows.append({"catchment_miles": miles, "tracts": len(underserved),
                 "deaths": underserved["overdose_deaths"].sum(),
                 "median_pct_black": underserved["pct_nh_black"].median(),
                 "median_pct_no_vehicle": underserved["pct_no_vehicle"].median()})

sensitivity = pd.DataFrame(rows)
baseline = underserved_by_catchment[2.0]
sensitivity["overlap_with_2_mile_group"] = [len(underserved_by_catchment[m] & baseline) / len(baseline)
                                            for m in CATCHMENT_SENSITIVITY_MILES]
sensitivity.round(2)

In [ ]:
# Same question with only Medicaid-accepting sites as supply. Most people in the
# underserved tracts would need a clinic that takes Medicaid.
medicaid_gap = tract_table.groupby("access_group")[["access_moud_per_100_deaths",
                                                    "access_moud_medicaid_per_100_deaths"]].median()
medicaid_gap.loc[group_order].round(2)

## What this shows

- **Distance is misleading here.** The highest-burden quartile of tracts has a median of 0.6 miles to the nearest MOUD site, closer than any other quartile, and the most sites per resident (3.5 per 100k vs about 1.6). Measured per overdose death, the same tracts have the *least* treatment supply (about 5 sites per 100 annual deaths vs 8.2 in the lowest-burden quartile).
- **227 tracts hold 12.4% of residents and 39.3% of overdose deaths** while falling below the county's treatment-per-death benchmark. They aren't deserts by distance (median 0.6 miles to a clinic), but they have 4.2 sites per 100 annual deaths, compared with 9.8 in high-burden tracts that are better served.
- **Who lives there.** Compared with better-served high-burden tracts, the underserved ones have lower median household income ($43,951 vs $58,189), more poverty (26% vs 19%), and more households without a vehicle (30% vs 21%). Their median tract is 80% Black. Uninsured rates are about the same (10%), so insurance status doesn't separate the two groups.
- **Medicaid narrows it further.** Counting only sites that accept Medicaid, the underserved tracts drop to 3.7 sites per 100 deaths, vs 8.3 in better-served high-burden tracts.
- **Holds up to the catchment choice.** At 1, 2, and 3 miles the underserved group has 242, 227, and 244 tracts, with 84 to 85% overlap and nearly identical demographics.

The main caveat is capacity: every site counts once, whether it treats 20 patients or 800. The next step is transit travel time as the catchment, since 30% of households in these tracts don't have a car. `04_model.ipynb` looks at which structural conditions go with higher overdose rates.